# Class 28: Agentic Workflows with CrewAI

এই notebook সম্পূর্ণ **self-contained** — project folder লাগে না। Section 1-2, 6-7 LLM ছাড়াই চলে; Section 3-5 Ollama (`llama3.1:8b`) লাগে।

| Section | কী শিখব |
| --- | --- |
| 1 | Agent / Task / Crew — fake LLM দিয়ে mechanics |
| 2 | Guardrail — retry, আর annotation-এর ফাঁদ |
| 3 | Live: এক agent-এর crew |
| 4 | Live: llama3.1 কি CrewAI-র ভেতরে সত্যিই tool ডাকে? |
| 5 | Live: grounded দুই-agent crew + code-এ citation যাচাই |
| 6 | Flow: `@start` → `@listen` → `@router`, capped retry |
| 7 | যে default-গুলো data বাইরে পাঠাতে পারে |

In [1]:
import os
# Telemetry is ON by default in CrewAI; these must be set BEFORE importing crewai.
os.environ.update(CREWAI_DISABLE_TELEMETRY="true", OTEL_SDK_DISABLED="true", CREWAI_TRACING_ENABLED="false")

import logging, re, time
from typing import Any, Tuple

import crewai
from crewai import LLM, Agent, Crew, Process, Task
from crewai.flow.flow import Flow, listen, router, start
from crewai.llms.base_llm import BaseLLM
from crewai.tools import BaseTool
from pydantic import BaseModel, Field, ValidationError

logging.getLogger("crewai").setLevel(logging.ERROR)
print("crewai", crewai.__version__)
OLLAMA = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.1, seed=42)
print(type(OLLAMA).__name__, "->", OLLAMA.base_url, "(native OpenAI-compatible client, no LiteLLM)")

crewai 1.15.29
OpenAICompatibleCompletion -> http://localhost:11434/v1 (native OpenAI-compatible client, no LiteLLM)


## 1. Agent / Task / Crew — fake LLM দিয়ে

একটা scripted `BaseLLM` CrewAI-র ReAct format-এ উত্তর দেয় ("Thought: ... Final Answer: ...")। এতে দেখা যায় `{topic}` কীভাবে বসে আর `context` দিয়ে প্রথম task-এর output দ্বিতীয় task-এর prompt-এ কীভাবে যায় — Ollama ছাড়াই।

> **Jupyter-এর ফাঁদ:** CrewAI 1.15-এ notebook-এর ভেতরে `crew.kickoff()` চালালে `RuntimeError: Agent execution was invoked synchronously from within a running event loop` — Jupyter নিজেই একটা event loop চালায়। Notebook-এ তাই `await crew.kickoff_async()`; সাধারণ script-এ `crew.kickoff()`। (Flow-এর `kickoff()` notebook-এও চলে — Flow ধাপগুলো আলাদা thread-এ চালায়।)

In [2]:
class FakeLLM(BaseLLM):
    def __init__(self, answers):
        super().__init__(model="fake")
        self.answers, self.prompts = list(answers), []
    def call(self, messages, *args, **kwargs):
        self.prompts.append(messages if isinstance(messages, str) else messages[-1]["content"])
        answer = self.answers.pop(0) if len(self.answers) > 1 else self.answers[0]
        return f"Thought: I now know the final answer\nFinal Answer: {answer}"
    def supports_function_calling(self):
        return False

fake = FakeLLM(["- Ollama default temperature is 0.8", "Report: set temperature explicitly."])
researcher = Agent(role="Researcher", goal="Find facts about {topic}", backstory="Careful.", llm=fake)
writer = Agent(role="Writer", goal="Write about {topic}", backstory="Clear.", llm=fake)
t1 = Task(description="List facts about {topic}.", expected_output="Bullets.", agent=researcher)
t2 = Task(description="Write one sentence about {topic} from the facts.", expected_output="One sentence.",
          agent=writer, context=[t1])
out = await Crew(agents=[researcher, writer], tasks=[t1, t2], process=Process.sequential).kickoff_async(inputs={"topic": "Ollama defaults"})
print("final:", out.raw)
print("task outputs:", [o.raw for o in out.tasks_output])
print("\nwriter's prompt contains the researcher's facts?", "temperature is 0.8" in str(fake.prompts[1]))

final: Report: set temperature explicitly.
task outputs: ['- Ollama default temperature is 0.8', 'Report: set temperature explicitly.']

writer's prompt contains the researcher's facts? True


## 2. Guardrail

Guardrail = output যাচাইয়ের function, `(True, value)` বা `(False, কারণ)`। False হলে কারণ সহ task আবার চলে। প্রথম উত্তর citation ছাড়া — guardrail ফেরত পাঠায়, দ্বিতীয়টা পাস। পরের cell-এ একটা ফাঁদ: `from __future__ import annotations` থাকলে return annotation string হয়ে যায়, আর CrewAI সেটা reject করে — আমাদের project-এ ঠিক এটাই ঘটেছিল।

In [3]:
def needs_citation(output):                       # NOTE: no return annotation (see next cell)
    if not re.search(r"\[[a-z_]+:\d+\]", output.raw):
        return False, "Every fact must end with a citation id like [doc:1]."
    return True, output.raw                       # return .raw, not the TaskOutput object

fake = FakeLLM(["Ollama is fast.", "Ollama's default temperature is 0.8 [ollama:1]."])
a = Agent(role="Researcher", goal="Cite facts", backstory="Careful.", llm=fake)
t = Task(description="One fact about Ollama.", expected_output="A cited fact.", agent=a,
         guardrail=needs_citation, guardrail_max_retries=2)
out = await Crew(agents=[a], tasks=[t]).kickoff_async()
print("result:", out.raw, "| LLM calls:", len(fake.prompts), "(first answer rejected)")

result: Ollama's default temperature is 0.8 [ollama:1]. | LLM calls: 2 (first answer rejected)


In [4]:
# The annotation trap: CrewAI validates a guardrail's return annotation. With `from __future__ import annotations`
# at the top of a file (common in real projects) every annotation becomes a STRING - we simulate that here.
def plain(output) -> tuple[bool, str]:
    return True, output.raw

def as_in_future_import(output) -> tuple[bool, str]:
    return True, output.raw
as_in_future_import.__annotations__["return"] = "tuple[bool, str]"   # what the __future__ import does

def no_annotation(output):
    return True, output.raw

for fn in (plain, as_in_future_import, no_annotation):
    try:
        Task(description="x", expected_output="y", agent=a, guardrail=fn)
        print(f"{fn.__name__}: accepted")
    except ValidationError as e:
        print(f"{fn.__name__}: ValidationError -> {e.errors()[0]['msg'][:70]}")

plain: accepted
as_in_future_import: ValidationError -> Value error, If return type is annotated, it must be Tuple[bool, Any]
no_annotation: accepted


## 3. Live — এক agent-এর crew

এবার আসল `llama3.1:8b`। CrewAI প্রতিটা agent-কে একটা ReAct loop হিসেবে চালায়; আমরা সময় আর token দেখি।

In [5]:
summarizer = Agent(role="Technical Summarizer", goal="Summarize text for engineers in one sentence.",
                   backstory="You are concise and never add facts.", llm=OLLAMA, max_iter=4)
text = ("Ollama's default sampling temperature is 0.8, which is too random for agents. "
        "By default it processes one request at a time per model, and its context window defaults to 4096 tokens.")
task = Task(description=f"Summarize this in one sentence:\n{text}", expected_output="One sentence.", agent=summarizer)
t0 = time.time()
out = await Crew(agents=[summarizer], tasks=[task]).kickoff_async()
print(out.raw, f"\n\n{time.time() - t0:.1f}s | tokens:", out.token_usage.total_tokens, "| requests:", out.token_usage.successful_requests)

Ollama's default sampling temperature is 0.8, which is too random for agents, and it processes one request at a time per model with a context window of 4096 tokens. 

2.2s | tokens: 174 | requests: 1


## 4. Live — tool কি সত্যিই ডাকা হয়?

একটা ছোট tool যেটা গুনে রাখে কয়বার ডাকা হলো, আর একটা data যেটা model আগে থেকে জানতে পারে না (বানানো "warehouse stock")। Agent যদি tool না ডেকে সংখ্যা বলে — সেটা বানানো।

In [6]:
STOCK = {"widget-a": 17, "widget-b": 0, "gadget-x": 342}
tool_calls = []

class StockInput(BaseModel):
    item: str = Field(..., description="Item code, e.g. widget-a")

class StockTool(BaseTool):
    name: str = "check_stock"
    description: str = "Return the current warehouse stock for an item code."
    args_schema: type[BaseModel] = StockInput
    def _run(self, item: str) -> str:
        tool_calls.append(item)
        return f"{item}: {STOCK.get(item.strip().lower(), 'UNKNOWN_ITEM')} units"

clerk = Agent(role="Warehouse Clerk", goal="Answer stock questions using the check_stock tool.",
              backstory="You always look up stock; you never guess.", tools=[StockTool()], llm=OLLAMA, max_iter=5)
q = Task(description="How many units of gadget-x and widget-b are in stock?",
         expected_output="The two stock numbers.", agent=clerk)
out = await Crew(agents=[clerk], tasks=[q]).kickoff_async()
print("answer:", out.raw)
print("tool was called", len(tool_calls), "time(s):", tool_calls, "| truth: gadget-x=342, widget-b=0")

answer: The two stock numbers are: gadget-x: 342 units, widget-b: 0 units.
tool was called 2 time(s): ['gadget-x', 'widget-b'] | truth: gadget-x=342, widget-b=0


**আমাদের run-এ:** এই সহজ, এক-tool, ছোট-prompt case-এ llama3.1:8b ঠিকমতো দুবার tool ডেকেছে আর সঠিক সংখ্যা দিয়েছে। কিন্তু project-এর researcher-এ (লম্বা task description, অনেক নিয়ম) একই model `search_corpus` একবারও ডাকেনি — নিজেই "search result" আর citation id বানিয়েছিল। অর্থাৎ ছোট model-এর tool use prompt আর task-এর ওপর নির্ভর করে — guarantee না। তাই নিয়ম: **যে কাজ code পারে (retrieval, lookup), code করুক** — data সরাসরি task-এ দিন, আর tool call সংখ্যা সবসময় log করুন।

## 5. Live — Grounded দুই-agent crew + code verification

ছোট একটা inline corpus। Data code থেকে task-এ বসে; Writer লেখে, Reviewer notes মিলিয়ে ঠিক করে; শেষে code প্রতিটা citation যাচাই করে।

In [7]:
CORPUS = {
    "ollama:1": "Ollama's default sampling temperature is 0.8, which is too random for agents and tool use.",
    "ollama:2": "By default Ollama processes one request at a time per model, so parallel tasks wait in a queue.",
    "crew:1": "CrewAI telemetry is on by default and needs three environment flags to switch off.",
    "crew:2": "A CrewAI guardrail returns (True, value) to accept output or (False, feedback) to retry the task.",
}
passages = "\n".join(f"[{k}] {v}" for k, v in CORPUS.items())

writer = Agent(role="Technical Writer", goal="Write short cited notes.",
               backstory="You only use the given passages and cite each fact.", llm=OLLAMA, max_iter=4)
reviewer = Agent(role="Fact-Checker", goal="Remove any sentence the passages do not support.",
                 backstory="Strict editor; you never add facts.", llm=OLLAMA, max_iter=4)
draft = Task(description=(f"Using ONLY these passages, write 3-4 sentences on running agents locally. "
                          f"End each sentence with its id in brackets.\n{passages}"),
             expected_output="3-4 cited sentences.", agent=writer, guardrail=needs_citation)
review = Task(description=(f"Check the draft against these passages and return only the corrected text, ids kept.\n{passages}"),
              expected_output="The corrected cited text.", agent=reviewer, context=[draft])
t0 = time.time()
out = await Crew(agents=[writer, reviewer], tasks=[draft, review]).kickoff_async()
print(out.raw, f"\n\n{time.time() - t0:.1f}s")

Here is the corrected text with the actual complete content:

To run agents locally, it is recommended to adjust the sampling temperature to a lower value, such as 0.8 [ollama:1]. This is because the default temperature is too random for agents and tool use. Agents can be run in parallel, but by default, Ollama processes one request at a time per model, so parallel tasks wait in a queue [ollama:2]. 

8.5s


In [8]:
def words(s):
    return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if len(w) > 2}

rows = []
for sent in re.split(r"(?<=[.!?])\s+", out.raw.strip()):
    ids = re.findall(r"\[([a-z]+:\d+)\]", sent)
    if not ids:
        rows.append(("uncited", sent[:80])); continue
    if any(i not in CORPUS for i in ids):
        rows.append(("INVALID ID", sent[:80])); continue
    overlap = len(words(re.sub(r"\[.*?\]", "", sent)) & words(" ".join(CORPUS[i] for i in ids)))
    support = overlap / max(len(words(re.sub(r"\[.*?\]", "", sent))), 1)
    rows.append(("supported" if support >= 0.45 else "weak", f"{support:.2f}  {sent[:80]}"))
for verdict, text in rows:
    print(f"{verdict:<11} {text}")

weak        0.17  Here is the corrected text with the actual complete content:

To run agents loca
uncited     This is because the default temperature is too random for agents and tool use.
supported   0.75  Agents can be run in parallel, but by default, Ollama processes one request at a


**আমাদের run-এ code কী ধরল:** Reviewer লিখেছে "temperature কমিয়ে ... such as 0.8 রাখুন" — অথচ source বলে 0.8-ই default আর সেটাই বেশি random। Reviewer নিজেই একটা ভুল ঢুকিয়েছে, আর code-এর support check সেই বাক্যকে `weak` (০.১৭) দেখিয়েছে। এছাড়া Reviewer "Here is the corrected text..." preamble যোগ করেছে — নির্দেশ মানেনি। এই দুটোই MAST-এর পরিচিত failure: ভুল verification আর task spec না মানা। এজন্যই শেষ gate code-এ।

## 6. Flow — deterministic shell

Flow-এর ধাপগুলো code-এ বাঁধা: `@start` → `@listen` → `@router` → label অনুযায়ী `@listen`। Retry-র hard cap plain Python loop-এ। এখানে "crew"-এর জায়গায় একটা নকল function — mechanics দেখার জন্য।

In [9]:
class State(BaseModel):
    attempts: int = 0
    report: str = ""
    outcome: str = ""

drafts = iter(["no citations here", "still nothing", "Temperature defaults to 0.8 [ollama:1]."])

class ReportFlow(Flow[State]):
    @start()
    def produce(self):
        for self.state.attempts in range(1, 3):            # hard cap: 2 rounds (MAST: "unaware of termination")
            self.state.report = next(drafts)               # stands in for crew.kickoff()
            if "[" in self.state.report:
                break

    @router(produce)
    def gate(self):
        return "ok" if "[" in self.state.report else "escalate"

    @listen("ok")
    def publish(self):
        self.state.outcome = "published"

    @listen("escalate")
    def to_human(self):
        self.state.outcome = "needs_human"

f = ReportFlow()
f.kickoff()
print(f.state)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ReportFlow                                                                                               │
│  ID: 82f30a71-1fd7-42fd-a1a0-c74408b9efeb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ReportFlow                                                                                               │
│  ID: 82f30a71-1fd7-42fd-a1a0-c74408b9efeb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: produce                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: produce                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: to_human                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: to_human                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ReportFlow                                                                                               │
│  ID: 82f30a71-1fd7-42fd-a1a0-c74408b9efeb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

attempts=2 report='still nothing' outcome='needs_human' id='82f30a71-1fd7-42fd-a1a0-c74408b9efeb'


দুই round-এর cap-এর ভেতরে citation আসেনি → `needs_human`। তৃতীয় draft (যেটায় citation আছে) কখনো চেষ্টাই হয়নি — cap ঠিক এটাই নিশ্চিত করে।

## 7. যে default-গুলো data বাইরে পাঠাতে পারে

Installed code-এর `model_fields` থেকে সরাসরি পড়ি — docs-এর চেয়ে বেশি নির্ভরযোগ্য।

In [10]:
for cls, names in ((Agent, ["max_iter", "allow_delegation", "cache", "respect_context_window"]),
                   (Task, ["guardrail_max_retries", "async_execution", "human_input"]),
                   (Crew, ["memory", "cache", "planning", "share_crew", "process"])):
    print(cls.__name__.ljust(6), {n: cls.model_fields[n].default for n in names})
print("\ntelemetry env:", {k: os.environ.get(k) for k in ("CREWAI_DISABLE_TELEMETRY", "OTEL_SDK_DISABLED", "CREWAI_TRACING_ENABLED")})

Agent  {'max_iter': 25, 'allow_delegation': False, 'cache': True, 'respect_context_window': True}
Task   {'guardrail_max_retries': 3, 'async_execution': False, 'human_input': False}
Crew   {'memory': False, 'cache': False, 'planning': False, 'share_crew': False, 'process': <Process.sequential: 'sequential'>}

telemetry env: {'CREWAI_DISABLE_TELEMETRY': 'true', 'OTEL_SDK_DISABLED': 'true', 'CREWAI_TRACING_ENABLED': 'false'}


`Agent.max_iter=25` ছোট model-এর জন্য অনেক বেশি — ৫-৮ রাখুন। `Crew.memory=False` রাখুন: চালু করলে embedder না দিলে default OpenAI embedder ব্যবহার হয় (run-time-এ key error বা data বাইরে)।

## 🧠 Brain Teasers

1. Section 4 তিনবার চালান। Tool call সংখ্যা কি প্রতিবার এক? উত্তরের সংখ্যা কি সঠিক?
2. Section 5-এ Reviewer-কে passages না দিয়ে শুধু draft দিন। Verification table কি খারাপ হয়?
3. Section 6-এ cap ৩ করুন। Outcome কী হয়? Cap ছাড়া (অসীম loop) production-এ কী বিপদ?
4. Section 2-এর guardrail `(True, output)` ফেরত দিলে আর task-এ `output_pydantic` থাকলে `out.pydantic` কী হয়? পরীক্ষা করে দেখুন।